# Perception scene dataset generation
#### Payload type: CPU + MuJoCo rendering + memory + disk I/O

In [ ]:
# Imports
import os
os.environ.setdefault('MUJOCO_GL', 'egl')
import cv2
import mujoco
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from matplotlib.patches import Ellipse
from IPython.display import HTML
import mediapy as media


In [ ]:
# Generate reproducible MuJoCo scenes for object-detection experiments
np.random.seed(0)

# Layout:
# - Three static occluders near the center.
# - A small static red cube (background distractor).
# - A moving red cylinder (target) behind the occluders.
# - A green box that is not part of the reference background image.
OCCLUDER_Y = 0.05
OCCLUDER_Z = 0.15
OCCLUDER_HALF_X = 0.030
OCCLUDER_HALF_Y = 0.030
OCCLUDER_HALF_Z = 0.10

LEFT_OCCLUDER_X = -0.18
MID_OCCLUDER_X = 0.00
RIGHT_OCCLUDER_X = 0.18

NOISE_CUBE_X = 0.33
NOISE_CUBE_Y = -0.08
NOISE_CUBE_Z = 0.015
NOISE_CUBE_HALF = 0.01

GREEN_BOX_X = 0.06
GREEN_BOX_Y = -0.13
GREEN_BOX_Z = 0.03
GREEN_BOX_HALF_X = 0.03
GREEN_BOX_HALF_Y = 0.03
GREEN_BOX_HALF_Z = 0.03

OBJECT_Z = 0.05
OBJECT_RADIUS = 0.025
OBJECT_HALF_HEIGHT = 0.060
TRACK_Y = 0.17  # larger y than occluders -> behind occluders from camera viewpoint
X_LEFT = -0.20
X_RIGHT = 0.50

BASE_LIGHT_POS = np.array([0.0, -0.15, 2.10], dtype=np.float64)
LIGHT_SWEEP_RADIUS_X = 0.90
LIGHT_SWEEP_RADIUS_Y = 0.55
LIGHT_SWEEP_HEIGHT_DELTA = 0.25


def moving_light_position(frame_idx, num_frames):
    phase = 2.0 * np.pi * frame_idx / max(1, 5 * (num_frames - 1))
    # Start at BASE_LIGHT_POS, then move the light to create shadow motion over time.
    x = BASE_LIGHT_POS[0] + LIGHT_SWEEP_RADIUS_X * (np.cos(phase) - 1.0)
    y = BASE_LIGHT_POS[1] + LIGHT_SWEEP_RADIUS_Y * np.sin(phase)
    z = BASE_LIGHT_POS[2] + LIGHT_SWEEP_HEIGHT_DELTA * np.sin(2.0 * phase)
    return np.array([x, y, z], dtype=np.float64)


def get_detection_scene_xml(include_red_object=True, include_green_object=False, light_pos=BASE_LIGHT_POS):
    red_object_block = f"""
        <body name='red_obj' pos='{X_LEFT} {TRACK_Y} {OBJECT_Z}'>
          <joint name='red_x' type='slide' axis='1 0 0'/>
          <joint name='red_y' type='slide' axis='0 1 0'/>
          <geom type='cylinder' size='{OBJECT_RADIUS} {OBJECT_HALF_HEIGHT}' rgba='0.95 0.10 0.10 1'/>
        </body>
    """ if include_red_object else ""

    green_object_block = f"""
        <geom name='green_box_distractor' type='box'
              pos='{GREEN_BOX_X} {GREEN_BOX_Y} {GREEN_BOX_Z}'
              size='{GREEN_BOX_HALF_X} {GREEN_BOX_HALF_Y} {GREEN_BOX_HALF_Z}'
              rgba='0.10 0.85 0.15 1'/>
    """ if include_green_object else ""

    lx, ly, lz = light_pos
    return f"""
    <mujoco model='single_object_tracking'>
      <option timestep='0.02' gravity='0 0 -9.81'/>
      <worldbody>
        <light name='scene_light' pos='{lx} {ly} {lz}' diffuse='0.75 0.75 0.75' castshadow='true'/>
        <camera name='oblique_top' pos='0 -0.50 0.50' xyaxes='1 0 0 0 0.611 0.792' fovy='50'/>
        <geom name='floor' type='plane' pos='0 0 0' size='1.0 1.0 0.1' rgba='0.95 0.95 0.95 1'/>
        <geom name='occluder_left' type='box' pos='{LEFT_OCCLUDER_X} {OCCLUDER_Y} {OCCLUDER_Z}' size='{OCCLUDER_HALF_X} {OCCLUDER_HALF_Y} {OCCLUDER_HALF_Z}' rgba='0.55 0.55 0.55 1'/>
        <geom name='occluder_mid' type='box' pos='{MID_OCCLUDER_X} {OCCLUDER_Y} {OCCLUDER_Z}' size='{OCCLUDER_HALF_X} {OCCLUDER_HALF_Y} {OCCLUDER_HALF_Z}' rgba='0.55 0.55 0.55 1'/>
        <geom name='occluder_right' type='box' pos='{RIGHT_OCCLUDER_X} {OCCLUDER_Y} {OCCLUDER_Z}' size='{OCCLUDER_HALF_X} {OCCLUDER_HALF_Y} {OCCLUDER_HALF_Z}' rgba='0.55 0.55 0.55 1'/>
        <geom name='red_cube_distractor' type='box' pos='{NOISE_CUBE_X} {NOISE_CUBE_Y} {NOISE_CUBE_Z}' size='{NOISE_CUBE_HALF} {NOISE_CUBE_HALF} {NOISE_CUBE_HALF}' rgba='0.90 0.12 0.12 1'/>
        {green_object_block}
        {red_object_block}
      </worldbody>
    </mujoco>
    """


def save_video_cv2(frames_rgb, video_path, fps=30):
    h, w = frames_rgb[0].shape[:2]
    writer = cv2.VideoWriter(video_path, cv2.VideoWriter_fourcc(*'mp4v'), fps, (w, h))
    final_path = video_path

    if not writer.isOpened():
        fallback_path = os.path.splitext(video_path)[0] + '.avi'
        writer = cv2.VideoWriter(fallback_path, cv2.VideoWriter_fourcc(*'MJPG'), fps, (w, h))
        final_path = fallback_path

    if not writer.isOpened():
        raise RuntimeError('Could not open OpenCV VideoWriter for MP4 or AVI output.')

    for frame_rgb in frames_rgb:
        writer.write(cv2.cvtColor(frame_rgb, cv2.COLOR_RGB2BGR))
    writer.release()
    return final_path


def render_background_frame(width, height):
    bg_model = mujoco.MjModel.from_xml_string(
        get_detection_scene_xml(include_red_object=False, include_green_object=False, light_pos=BASE_LIGHT_POS)
    )
    bg_data = mujoco.MjData(bg_model)
    bg_renderer = mujoco.Renderer(bg_model, width=width, height=height)
    mujoco.mj_forward(bg_model, bg_data)
    bg_renderer.update_scene(bg_data, camera='oblique_top')
    return bg_renderer.render().copy()


def generate_scene_dataset(
    num_frames=180,
    fps=30,
    width=640,
    height=480,
    include_green_object=False,
    apply_lighting_variation=False,
    out_dir='object_detection_data/scene',
    prefix='scene',
):
    model = mujoco.MjModel.from_xml_string(
        get_detection_scene_xml(
            include_red_object=True,
            include_green_object=include_green_object,
            light_pos=BASE_LIGHT_POS,
        )
    )
    data = mujoco.MjData(model)
    renderer = mujoco.Renderer(model, width=width, height=height)

    os.makedirs(out_dir, exist_ok=True)
    video_path = os.path.join(out_dir, f'{prefix}_scene.mp4')
    gt_path = os.path.join(out_dir, f'{prefix}_gt.npz')
    bg_path = os.path.join(out_dir, f'{prefix}_background.png')

    background_rgb = render_background_frame(width, height)
    cv2.imwrite(bg_path, cv2.cvtColor(background_rgb, cv2.COLOR_RGB2BGR))

    red_x_id = mujoco.mj_name2id(model, mujoco.mjtObj.mjOBJ_JOINT, 'red_x')
    red_y_id = mujoco.mj_name2id(model, mujoco.mjtObj.mjOBJ_JOINT, 'red_y')
    red_body_id = mujoco.mj_name2id(model, mujoco.mjtObj.mjOBJ_BODY, 'red_obj')
    light_id = mujoco.mj_name2id(model, mujoco.mjtObj.mjOBJ_LIGHT, 'scene_light')
    red_x_qpos = model.jnt_qposadr[red_x_id]
    red_y_qpos = model.jnt_qposadr[red_y_id]

    frames_rgb = []
    times = np.arange(num_frames) / fps
    world_xy = np.zeros((num_frames, 2), dtype=np.float32)

    def smoothstep(a):
        return a * a * (3.0 - 2.0 * a)

    half = num_frames // 2
    for i in range(num_frames):
        if i < half:
            alpha = smoothstep(i / max(1, half - 1))
            x = X_LEFT + alpha * (X_RIGHT - X_LEFT)
        else:
            alpha = smoothstep((i - half) / max(1, num_frames - half - 1))
            x = X_RIGHT - alpha * (X_RIGHT - X_LEFT)

        model.light_pos[light_id] = moving_light_position(i, num_frames) if apply_lighting_variation else BASE_LIGHT_POS
        data.qpos[red_x_qpos] = x
        data.qpos[red_y_qpos] = TRACK_Y

        mujoco.mj_forward(model, data)
        world_xy[i] = data.xpos[red_body_id, :2]
        renderer.update_scene(data, camera='oblique_top')
        frames_rgb.append(renderer.render().copy())

    saved_video_path = save_video_cv2(frames_rgb, video_path, fps=fps)
    np.savez(gt_path, times=times, world_xy=world_xy, color_names=np.array(['red']))
    return frames_rgb, background_rgb, saved_video_path, gt_path, bg_path


SCENE_CONFIGS = {
    'static_light': dict(include_green_object=False, apply_lighting_variation=False),
    'dynamic_light': dict(include_green_object=True, apply_lighting_variation=True),
}

scene_data = {}
for scene_name, scene_cfg in SCENE_CONFIGS.items():
    frames, background, video_path, gt_path, bg_path = generate_scene_dataset(
        out_dir=f'object_detection_data/{scene_name}',
        prefix=scene_name,
        **scene_cfg,
    )
    scene_data[scene_name] = {
        'frames': frames,
        'background': background,
        'video_path': video_path,
        'gt_path': gt_path,
        'background_path': bg_path,
    }
    print(f'{scene_name}:')
    print(f'  video: {video_path}')
    print(f'  background image: {bg_path}')
    print(f'  ground truth: {gt_path}')

static_scene = scene_data['static_light']
dynamic_scene = scene_data['dynamic_light']

frames_static_rgb = static_scene['frames']
background_static_rgb = static_scene['background']
frames_dynamic_rgb = dynamic_scene['frames']
background_dynamic_rgb = dynamic_scene['background']
scene_dynamic_gt_path = dynamic_scene['gt_path']

fig, axes = plt.subplots(2, 2, figsize=(10, 8))
for row, scene_name in enumerate(['static_light', 'dynamic_light']):
    scene = scene_data[scene_name]
    axes[row, 0].imshow(scene['background'])
    axes[row, 0].set_title(f'{scene_name} background')
    axes[row, 0].axis('off')
    axes[row, 1].imshow(scene['frames'][40])
    axes[row, 1].set_title(f'{scene_name} sample frame')
    axes[row, 1].axis('off')

plt.tight_layout()
plt.show()

for scene_name in ['static_light', 'dynamic_light']:
    print(f'{scene_name} video preview:')
    media.show_video(scene_data[scene_name]['frames'], fps=30, loop=False)
